# Fase 2B — Machine Learning: Hard Voting LR + RF + XGBoost (IroSvA) — v2

**Qué es la v2.** Continuación de la v1 (`ML_HardVoting_TFIDF10k_GroupNestedCV_5x3.ipynb`). La regla no cambia (mayoría 2 de 3, fijada a priori) ni la Nested CV (la de la v1). Lo nuevo: el test se evalúa con las **5 semillas** de `ML_TFIDF_10k_GroupNestedCV_5x3_v2.ipynb` — en cada semilla *s* votan LR, RF y XGBoost de la semilla *s* — y se reporta media ± desviación, igual que los Transformers. La semilla 42 debe reproducir la votación de la v1.

No entrena nada: lee `predicciones_test_ml_5_semillas.csv` de la v2. Requiere haber ejecutado antes el test de la v2. Resultados en `../data/resultados_ml_hard_voting_tfidf10k_v2/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score, accuracy_score

DATA_DIR = Path("../data")
ML_V2_DIR = DATA_DIR / "resultados_ml_groupcv_tfidf_lr_rf_10k_v2"
HV_V1_DIR = DATA_DIR / "resultados_ml_hard_voting_tfidf10k"
OUT_DIR = DATA_DIR / "resultados_ml_hard_voting_tfidf10k_v2"
OUT_DIR.mkdir(parents=True, exist_ok=True)
VARIANTES = ["mx", "es", "cu"]
MODELOS = ["LR", "RF", "XGBoost"]

EJECUTAR_TEST_OFICIAL = False
CONFIRMACION_TEST = ""

ruta = ML_V2_DIR / "predicciones_test_ml_5_semillas.csv"
if not ruta.exists():
    raise FileNotFoundError(f"Falta {ruta}: ejecuta antes el test (sección 9) de ML_TFIDF_10k_GroupNestedCV_5x3_v2.ipynb.")
print("Entrada:", ruta.resolve())

## Votación por semilla (test oficial — bloqueado)

Para cada variante y semilla se alinean por `ID` las predicciones de LR, RF y XGBoost (se verifica que las etiquetas coincidan) y gana la clase votada por al menos 2 de los 3. Se reporta F1-Macro, F1 de la clase irónica y F1-Macro sin los textos que coinciden con el train (mismo conjunto común de la comparación final).

In [ ]:
if not (EJECUTAR_TEST_OFICIAL and CONFIRMACION_TEST == "TEST_OFICIAL_CONGELADO"):
    print("TEST OFICIAL BLOQUEADO.")
else:
    pred_ml = pd.read_csv(ruta, dtype={"ID": str})
    ruta_coinc = DATA_DIR / "resultados_comparacion_final_v6" / "ids_test_coincidentes_con_train.csv"
    coinc = pd.read_csv(ruta_coinc, dtype={"ID": str}) if ruta_coinc.exists() else pd.DataFrame(columns=["variante", "ID"])
    filas, predicciones = [], []
    for variante in VARIANTES:
        ids_coinc = set(coinc.loc[coinc["variante"] == variante, "ID"].astype(str))
        for seed in sorted(pred_ml["seed"].unique()):
            sub = pred_ml[(pred_ml["variante"] == variante) & (pred_ml["seed"] == seed)]
            w = sub.pivot(index="ID", columns="modelo", values=["y_pred", "y_true"])
            assert len(w) == 600 and set(MODELOS) <= set(w["y_pred"].columns), f"{variante}/{seed}: faltan modelos o textos"
            assert w["y_true"][MODELOS].nunique(axis=1).eq(1).all(), f"{variante}/{seed}: etiquetas desalineadas"
            y = w["y_true"][MODELOS[0]].astype(int).to_numpy()
            pred = (w["y_pred"][MODELOS].astype(int).sum(axis=1).to_numpy() >= 2).astype(int)
            limpio = ~w.index.isin(list(ids_coinc))
            predicciones.append(pd.DataFrame({"variante": variante, "modelo": "HardVoting", "seed": seed,
                                              "ID": w.index.astype(str), "y_true": y, "y_pred": pred}))
            filas.append({"variante": variante, "modelo": "HardVoting", "seed": seed,
                          "f1_macro": f1_score(y, pred, average="macro"),
                          "f1_ironia": f1_score(y, pred, pos_label=1, zero_division=0),
                          "f1_macro_sin_coincidencias": f1_score(y[limpio], pred[limpio], average="macro"),
                          "accuracy": accuracy_score(y, pred)})
    RES = pd.DataFrame(filas)
    PRED = pd.concat(predicciones, ignore_index=True)
    RES.to_csv(OUT_DIR / "hard_voting_test_5_semillas.csv", index=False)
    PRED.to_csv(OUT_DIR / "hard_voting_test_predicciones_5_semillas.csv", index=False)
    resumen = RES.groupby("variante").agg(f1_macro_mean=("f1_macro", "mean"), f1_macro_std=("f1_macro", "std"),
                                          f1_ironia_mean=("f1_ironia", "mean"),
                                          f1_sin_coinc_mean=("f1_macro_sin_coincidencias", "mean")).reset_index()
    resumen.to_csv(OUT_DIR / "hard_voting_test_resumen.csv", index=False)
    display(RES.pivot(index="variante", columns="seed", values="f1_macro").round(4))
    display(resumen.round(4))
    ruta_v1 = HV_V1_DIR / "hard_voting_test_predicciones.csv"
    if ruta_v1.exists():
        v1 = pd.read_csv(ruta_v1, dtype={"ID": str}).set_index(["variante", "ID"])["y_pred"]
        s42 = PRED[PRED["seed"] == 42].set_index(["variante", "ID"])["y_pred"]
        acuerdo = (s42 == v1.reindex(s42.index)).groupby(level=0).mean()
        print("Acuerdo de la semilla 42 con la votación de la v1 (debe ser 1.0):", acuerdo.round(4).to_dict())

## Resumen

- Misma regla que la v1 (mayoría 2 de 3, sin pesos ni selección de subconjuntos).
- 5 semillas, las mismas de los Transformers; se reporta media ± desviación. La Nested CV de referencia es la de la v1.
- Si la votación no supera a los modelos individuales, se reporta tal cual.